# Dataset Benchmarking

This notebook is a behind-the-scenes benchmarking notebook, mainly for use by developers. The recommended way for users to interact with the dataset is via the `Measurement` object and its associated context manager. See the corresponding notebook for a comprehensive toturial on how to use those.

In [1]:
%matplotlib inline
from pathlib import Path

import numpy as np

import qcodes as qc
from qcodes.dataset import (
    ParamSpec,
    initialise_or_create_database_at,
    load_or_create_experiment,
    new_data_set,
)

Logging hadn't been started.
Activating auto-logging. Current session state plus future input saved.
Filename       : /home/runner/.qcodes/logs/command_history.log
Mode           : append
Output logging : True
Raw input log  : False
Timestamping   : True
State          : active


Qcodes Logfile : /home/runner/.qcodes/logs/250310-17634-qcodes.log


In [2]:
qc.config.core.db_location

'~/experiments.db'

In [3]:
initialise_or_create_database_at(Path.cwd() / "benchmarking.db")

## Setup

In [4]:
exp = load_or_create_experiment("benchmarking", sample_name="the sample is a lie")
exp

benchmarking#the sample is a lie#1@/home/runner/work/Qcodes/Qcodes/docs/examples/DataSet/benchmarking.db
--------------------------------------------------------------------------------------------------------

Now we can create a dataset. Note two things:

    - if we don't specfiy a exp_id, but we have an experiment in the experiment container the dataset will go into that one.
    - dataset can be created from the experiment object
    

In [5]:
dataSet = new_data_set("benchmark_data", exp_id=exp.exp_id)
exp

benchmarking#the sample is a lie#1@/home/runner/work/Qcodes/Qcodes/docs/examples/DataSet/benchmarking.db
--------------------------------------------------------------------------------------------------------
1-benchmark_data-1-None-0

In this benchmark we will assueme that we are doing a 2D loop and investigate the performance implications of writing to the dataset

In [6]:
x_shape = 100
y_shape = 100

## Baseline: Generate data

In [7]:
%%time
for x in range(x_shape):
    for y in range(y_shape):
        z = np.random.random_sample(1)

CPU times: user 10.8 ms, sys: 1.97 ms, total: 12.7 ms
Wall time: 12.5 ms


and store in memory

In [8]:
x_data = np.zeros((x_shape, y_shape))
y_data = np.zeros((x_shape, y_shape))
z_data = np.zeros((x_shape, y_shape))

In [9]:
%%time
for x in range(x_shape):
    for y in range(y_shape):
        x_data[x, y] = x
        y_data[x, y] = y
        z_data[x, y] = np.random.random_sample()

CPU times: user 8.99 ms, sys: 2 μs, total: 8.99 ms
Wall time: 8.94 ms


## Add to dataset inside double loop

In [10]:
double_dataset = new_data_set(
    "doubledata",
    exp_id=exp.exp_id,
    specs=[
        ParamSpec("x", "numeric"),
        ParamSpec("y", "numeric"),
        ParamSpec("z", "numeric"),
    ],
)
double_dataset.mark_started()

Note that this is so slow that we are only doing a 10th of the computation

In [11]:
%%time
for x in range(x_shape // 10):
    for y in range(y_shape):
        double_dataset.add_results([{"x": x, "y": y, "z": np.random.random_sample()}])

CPU times: user 187 ms, sys: 65.3 ms, total: 252 ms
Wall time: 712 ms


## Add the data in outer loop and store as np array

In [12]:
single_dataset = new_data_set(
    "singledata",
    exp_id=exp.exp_id,
    specs=[ParamSpec("x", "array"), ParamSpec("y", "array"), ParamSpec("z", "array")],
)
single_dataset.mark_started()
x_data = np.zeros(y_shape)
y_data = np.zeros(y_shape)
z_data = np.zeros(y_shape)

In [13]:
%%time
for x in range(x_shape):
    for y in range(y_shape):
        x_data[y] = x
        y_data[y] = y
        z_data[y] = np.random.random_sample(1)
    single_dataset.add_results([{"x": x_data, "y": y_data, "z": z_data}])

2025-03-10 13:32:35,416 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ <timed exec>:5: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)



CPU times: user 38.6 ms, sys: 4.86 ms, total: 43.5 ms
Wall time: 58.5 ms


## Save once after loop

In [14]:
zero_dataset = new_data_set(
    "zerodata",
    exp_id=exp.exp_id,
    specs=[ParamSpec("x", "array"), ParamSpec("y", "array"), ParamSpec("z", "array")],
)
zero_dataset.mark_started()
x_data = np.zeros((x_shape, y_shape))
y_data = np.zeros((x_shape, y_shape))
z_data = np.zeros((x_shape, y_shape))

In [15]:
%%time
for x in range(x_shape):
    for y in range(y_shape):
        x_data[x, y] = x
        y_data[x, y] = y
        z_data[x, y] = np.random.random_sample(1)
zero_dataset.add_results([{"x": x_data, "y": y_data, "z": z_data}])

2025-03-10 13:32:35,492 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ <timed exec>:5: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)



CPU times: user 25.4 ms, sys: 1.87 ms, total: 27.3 ms
Wall time: 27.4 ms


## Array parameter

In [16]:
array1D_dataset = new_data_set(
    "array1Ddata",
    exp_id=exp.exp_id,
    specs=[ParamSpec("x", "array"), ParamSpec("y", "array"), ParamSpec("z", "array")],
)
array1D_dataset.mark_started()
y_setpoints = np.arange(y_shape)

In [17]:
%%timeit
for x in range(x_shape):
    x_data[x, :] = x
    array1D_dataset.add_results(
        [{"x": x_data[x, :], "y": y_setpoints, "z": np.random.random_sample(y_shape)}]
    )

43.8 ms ± 2.4 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [18]:
x_data = np.zeros((x_shape, y_shape))
y_data = np.zeros((x_shape, y_shape))
z_data = np.zeros((x_shape, y_shape))
y_setpoints = np.arange(y_shape)

In [19]:
array0D_dataset = new_data_set(
    "array0Ddata",
    exp_id=exp.exp_id,
    specs=[ParamSpec("x", "array"), ParamSpec("y", "array"), ParamSpec("z", "array")],
)
array0D_dataset.mark_started()

In [20]:
%%timeit
for x in range(x_shape):
    x_data[x, :] = x
    y_data[x, :] = y_setpoints
    z_data[x, :] = np.random.random_sample(y_shape)
array0D_dataset.add_results([{"x": x_data, "y": y_data, "z": z_data}])

1.95 ms ± 65.7 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


## Insert many

In [21]:
data = []
for i in range(100):
    for j in range(100):
        data.append({"x": i, "y": j, "z": np.random.random_sample()})

In [22]:
many_Data = new_data_set(
    "many_data",
    exp_id=exp.exp_id,
    specs=[
        ParamSpec("x", "numeric"),
        ParamSpec("y", "numeric"),
        ParamSpec("z", "numeric"),
    ],
)
many_Data.mark_started()

In [23]:
%%timeit
many_Data.add_results(data)

19 ms ± 574 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
